<a href="https://colab.research.google.com/github/sumerahmughal/iat360-vegetable-recognition/blob/antony-epoch-experiments/notebooks/Antony_Epoch_Experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Person D – Epoch experiments + evaluation on our own photos
Baseline: yolo11n-cls.pt, epochs=15, imgsz=224, batch=16, seed=42. This notebook changes only the number of epochs (25, 75, 100), then evaluates the baseline and all 3 runs on the group's own 120 photos.

Runtime → Change runtime type → **T4 GPU** before running. Run cells top to bottom.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q ultralytics

Mounted at /content/drive


## Settings – check these two paths

In [3]:
PROJECT = "/content/drive/MyDrive/IAT 360 CV PROJECT"
ZIP_PATH = f"{PROJECT}/vegetable_dataset.zip"
OWN_PHOTOS_ROOT = PROJECT
RESULTS = f"{PROJECT}/training_results"
BASELINE_WEIGHTS = f"{RESULTS}/baseline/weights/best.pt"

CLASSES = ["Broccoli", "Capsicum", "Carrot", "Potato", "Tomato"]
BASE = dict(epochs=15, imgsz=224, seed=42)
BATCH_SIZE = 16
EPOCHS = [25, 75, 100]

## 1. Unzip the online dataset

In [4]:
import zipfile, os
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall("/content/")
DATA = "/content/vegetable_dataset"
for s in ["train","val","test"]:
    print(s, {c: len(os.listdir(f"{DATA}/{s}/{c}")) for c in sorted(os.listdir(f"{DATA}/{s}")) if os.path.isdir(f"{DATA}/{s}/{c}")})

train {'Broccoli': 1000, 'Capsicum': 1000, 'Carrot': 1000, 'Potato': 1000, 'Tomato': 1000}
val {'Broccoli': 200, 'Capsicum': 200, 'Carrot': 200, 'Potato': 200, 'Tomato': 200}
test {'Broccoli': 200, 'Capsicum': 200, 'Carrot': 200, 'Potato': 200, 'Tomato': 200}


## 2. Build the own-photo test set
Finds every photo named like `carrot_alyssa_003.jpeg` anywhere under `OWN_PHOTOS_ROOT` and sorts it into a class folder by the first word of its name.
Check the printed counts: you want ~24 per class (6 × 4 people) and 120 total. Anything listed as *skipped* has a name that didn't match – rename it in Drive and rerun.

In [5]:
import re, shutil, glob
from collections import Counter
OWN = "/content/own_test_ds"
shutil.rmtree(OWN, ignore_errors=True)
for c in CLASSES: os.makedirs(f"{OWN}/test/{c}", exist_ok=True)
os.symlink(f"{DATA}/train", f"{OWN}/train"); os.symlink(f"{DATA}/val", f"{OWN}/val")   # ultralytics wants these to exist

lookup = {c.lower(): c for c in CLASSES}
lookup.update({"pepper": "Capsicum", "bellpepper": "Capsicum", "tomatoes": "Tomato", "carrots": "Carrot", "potatoes": "Potato"})
pat = re.compile(r"^([a-z]+)_([a-z]+)_(\d+)", re.I)
counts, people, skipped, seen = Counter(), Counter(), [], set()
for p in glob.glob(f"{OWN_PHOTOS_ROOT}/**/*", recursive=True):
    if "vegetable_dataset" in p or "training_results" in p or not os.path.isfile(p): continue
    if not p.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        if p.lower().endswith((".heic", ".heif")): skipped.append(p + "   <- HEIC: convert to .jpg")
        continue
    name = os.path.basename(p); m = pat.match(name)
    if not m or m.group(1).lower() not in lookup: skipped.append(p); continue
    key = (m.group(1).lower(), m.group(2).lower(), int(m.group(3)))
    if key in seen: skipped.append(p + "   <- duplicate"); continue
    seen.add(key); cls = lookup[m.group(1).lower()]
    shutil.copy(p, f"{OWN}/test/{cls}/{m.group(1).lower()}_{m.group(2).lower()}_{int(m.group(3)):03d}.jpg")
    counts[cls] += 1; people[m.group(2).lower()] += 1
print("Per class:", dict(counts)); print("Per person:", dict(people)); print("Total:", sum(counts.values()))
print("Skipped:", *skipped[:40], sep="\n  ")

Per class: {'Potato': 24, 'Tomato': 24, 'Capsicum': 24, 'Carrot': 24, 'Broccoli': 24}
Per person: {'sumerah': 30, 'alyssa': 30, 'grace': 30, 'antony': 30}
Total: 120
Skipped:
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_001.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_003.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_004.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_002.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_005.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_sumerah_006.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original photos/Broccoli/broccoli_alyssa_003.jpeg   <- duplicate
  /content/drive/MyDrive/IAT 360 CV PROJECT/original pho

## 3. T3. Train the 3 epoch runs
Everything except epochs is identical to the baseline. Results save to Drive automatically.

In [12]:
import os

weights_path = os.path.join(
    REAL_RESULTS, "epochs25_final", "weights"
)

print("Folder exists:", os.path.exists(weights_path))
print("\nFiles saved:")

for file in os.listdir(weights_path):
    full_path = os.path.join(weights_path, file)
    print(file, round(os.path.getsize(full_path) / (1024 * 1024), 2), "MB")

Folder exists: True

Files saved:
best.pt 3.05 MB
last.pt 3.05 MB


In [11]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT = "/content/drive/MyDrive/IAT 360 CV PROJECT"
REAL_RESULTS = os.path.join(PROJECT, "training_results")

print("Results folder exists:", os.path.exists(REAL_RESULTS))
print("Folders:", os.listdir(REAL_RESULTS))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results folder exists: True
Folders: ['baseline', 'batch8', 'batch32', 'batch64', 'eval_online_baseline16', 'eval_own_baseline16', 'eval_online_batch8', 'eval_own_batch8', 'eval_online_batch32', 'eval_own_batch32', 'eval_online_batch64', 'eval_own_batch64', 'batch_size_comparison.csv', 'lr_0.001', 'lr_0.005', 'lr_0.02', 'eval_online_baseline', 'eval_own_baseline', 'eval_online_lr_0.001', 'eval_own_lr_0.001', 'eval_online_lr_0.005', 'eval_own_lr_0.005', 'eval_online_lr_0.02', 'eval_own_lr_0.02', 'learning_rate_comparison.csv', 'grace_imgsz_320', 'grace_imgsz_480', 'grace_imgsz_800', 'eval_online_224', 'eval_own_224', 'eval_online_320', 'eval_own_320', 'eval_online_480', 'eval_own_480', 'eval_online_800', 'eval_own_800', 'imgsz_comparison.csv', 'epochs25_final', 'epochs75_final']


In [ ]:
from ultralytics import YOLO
import os

# Experiment settings
e = 75
run_name = "epochs75_final"

# Exact output folder in Google Drive
save_path = os.path.join(REAL_RESULTS, run_name)

# Prevent accidental overwriting
if os.path.exists(save_path):
    raise FileExistsError(
        f"{save_path} already exists. "
        "Please check the folder before running again."
    )

# Change ONLY the number of epochs
settings = {**BASE, "epochs": e}

# Load the original pretrained YOLO model
model = YOLO("yolo11n-cls.pt")

# Train the model
model.train(
    data=DATA,
    batch=BATCH_SIZE,
    project=REAL_RESULTS,
    name=run_name,
    exist_ok=False,
    plots=False,
    **settings
)

Ultralytics 8.4.207 🚀 Python-3.13.18 torch-2.14.1+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/vegetable_dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=75, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=epochs75_final, nbs=64, nms=None, opset=None, optimize=Fal

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b222d2200c0>
curves: []
curves_results: []
fitness: 1.0
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 1.0, 'metrics/accuracy_top5': 1.0, 'fitness': 1.0}
save_dir: PosixPath('/content/drive/.shortcut-targets-by-id/13RmDr23rKxwQBszpCTxYbo_J7gasF5Ak/IAT 360 CV PROJECT/training_results/epochs75_final')
speed: {'preprocess': 0.0012875839966000058, 'inference': 10.775602255002013, 'loss': 5.404599869507365e-05, 'postprocess': 0.00026957299996865913}
top1: 1.0
top5: 1.0

In [ ]:
from glob import glob
from collections import defaultdict
from ultralytics import YOLO

model_75 = YOLO(model_path)

image_paths = []
true_labels = []

for vegetable in ["Broccoli", "Capsicum", "Carrot", "Potato", "Tomato"]:
    folder = os.path.join(photos_path, vegetable)

    for file in glob(os.path.join(folder, "*")):
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            image_paths.append(file)
            true_labels.append(vegetable.lower())

print("Total group photos:", len(image_paths))

predictions = model_75.predict(
    source=image_paths,
    imgsz=224,
    batch=16,
    verbose=False
)

correct = 0
class_results = defaultdict(lambda: [0, 0])

for true_label, result in zip(true_labels, predictions):
    predicted_label = result.names[result.probs.top1].lower()
    class_results[true_label][1] += 1

    if predicted_label == true_label:
        correct += 1
        class_results[true_label][0] += 1

print("\n75-EPOCH RESULTS")
print("Correct:", correct, "/ 120")
print("Accuracy:", round(correct / len(image_paths) * 100, 2), "%")

print("\nCLASS-WISE ACCURACY")
for vegetable, (right, total) in class_results.items():
    print(f"{vegetable.title()}: {right}/{total} ({right/total*100:.1f}%)")

Total group photos: 120


In [13]:
from ultralytics import YOLO
import os

e = 100
run_name = "epochs100_final"
save_path = os.path.join(REAL_RESULTS, run_name)

if os.path.exists(save_path):
    raise FileExistsError(
        f"{save_path} already exists. Check it before proceeding."
    )

settings = {**BASE, "epochs": e}

model = YOLO("yolo11n-cls.pt")

model.train(
    data=DATA,
    batch=BATCH_SIZE,
    project=REAL_RESULTS,
    name=run_name,
    exist_ok=False,
    plots=False,
    **settings
)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/vegetable_dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fre

ultralytics.utils.metrics.ClassifyMetrics object with attributes:

confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7efb841dc910>
curves: []
curves_results: []
fitness: 1.0
keys: ['metrics/accuracy_top1', 'metrics/accuracy_top5']
results_dict: {'metrics/accuracy_top1': 1.0, 'metrics/accuracy_top5': 1.0, 'fitness': 1.0}
save_dir: PosixPath('/content/drive/.shortcut-targets-by-id/13RmDr23rKxwQBszpCTxYbo_J7gasF5Ak/IAT 360 CV PROJECT/training_results/epochs100_final')
speed: {'preprocess': 0.08077492999927927, 'inference': 0.6627398090022325, 'loss': 0.0003724020007211948, 'postprocess': 0.0028077929991923156}
top1: 1.0
top5: 1.0

In [14]:
import os

weights_path = os.path.join(
    REAL_RESULTS, "epochs100_final", "weights"
)

print("Folder exists:", os.path.exists(weights_path))

for file in os.listdir(weights_path):
    full_path = os.path.join(weights_path, file)
    print(file, round(os.path.getsize(full_path)/(1024*1024), 2), "MB")

Folder exists: True
last.pt 3.05 MB
best.pt 3.05 MB


In [16]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT = "/content/drive/MyDrive/IAT 360 CV PROJECT"
REAL_RESULTS = os.path.join(PROJECT, "training_results")
photos_path = os.path.join(PROJECT, "original photos")

print("Photos folder exists:", os.path.exists(photos_path))
print("Training results folder exists:", os.path.exists(REAL_RESULTS))
print("Photo class folders:", os.listdir(photos_path))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Photos folder exists: True
Training results folder exists: True
Photo class folders: ['Broccoli', 'Capsicum', 'Carrot', 'Potato', 'Tomato']


In [ ]:
import os
import csv
from collections import defaultdict
from ultralytics import YOLO

# Model checkpoints
models_to_test = {
    "25 epochs": os.path.join(REAL_RESULTS, "epochs25_final", "weights", "best.pt"),
    "75 epochs": os.path.join(REAL_RESULTS, "epochs75_final", "weights", "best.pt"),
    "100 epochs": os.path.join(REAL_RESULTS, "epochs100_final", "weights", "best.pt")
}

# Collect all 120 photos, using their folder names as true labels
image_files = []

for class_name in sorted(os.listdir(photos_path)):
    class_folder = os.path.join(photos_path, class_name)

    if os.path.isdir(class_folder):
        for filename in sorted(os.listdir(class_folder)):
            if filename.lower().endswith((".jpg", ".jpeg", ".png")):
                image_files.append((
                    os.path.join(class_folder, filename),
                    class_name.lower()
                ))

print("Total photos:", len(image_files))

# Evaluate each model in batches of 10
all_results = {}

for model_name, checkpoint in models_to_test.items():

    print(f"\nEvaluating {model_name}...")

    model = YOLO(checkpoint)
    results_by_class = defaultdict(lambda: {"correct": 0, "total": 0})
    errors = []
    total_correct = 0

    for start in range(0, len(image_files), 10):
        batch = image_files[start:start + 10]
        paths = [item[0] for item in batch]

        predictions = model.predict(
            source=paths,
            imgsz=224,
            batch=10,
            verbose=False
        )

        for (path, true_label), prediction in zip(batch, predictions):
            predicted_label = str(model.names[prediction.probs.top1]).lower()

            results_by_class[true_label]["total"] += 1

            if predicted_label == true_label:
                results_by_class[true_label]["correct"] += 1
                total_correct += 1
            else:
                errors.append({
                    "image": os.path.basename(path),
                    "actual": true_label,
                    "predicted": predicted_label
                })

    accuracy = total_correct / len(image_files) * 100

    all_results[model_name] = {
        "accuracy": accuracy,
        "correct": total_correct,
        "classes": dict(results_by_class),
        "errors": errors
    }

    print(f"Accuracy: {total_correct}/{len(image_files)} = {accuracy:.1f}%")
    print("Class-wise results:")

    for class_name in sorted(results_by_class):
        r = results_by_class[class_name]
        print(f"  {class_name}: {r['correct']}/{r['total']}")

    print("Incorrect predictions:", len(errors))

    del model

print("\nALL THREE MODELS EVALUATED!")

Total photos: 120

Evaluating 25 epochs...


In [ ]:

from ultralytics import YOLO

for e in EPOCHS:
    if os.path.exists(f"{RESULTS}/epochs{e}/weights/best.pt"):
        print(f"epochs{e} already trained – skipping")
        continue

    settings = {**BASE, "epochs": e}

    YOLO("yolo11n-cls.pt").train(
        data=DATA,
        batch=BATCH_SIZE,
        project=RESULTS,
        name=f"epochs{e}",
        exist_ok=True,
        plots=True,
        **settings
    )


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/vegetable_dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze

## 4. Evaluate baseline + all runs on (a) the online test split and (b) our own photos
(b) is the number that matters for the report – it's the assignment's test set.

In [ ]:
import pandas as pd
runs = {"baseline (b=16)": BASELINE_WEIGHTS, **{f"batch {b}": f"{RESULTS}/batch{b}/weights/best.pt" for b in BATCH_SIZES}}
rows = []
for label, w in runs.items():
    m = YOLO(w)
    online = m.val(data=DATA, split="test", imgsz=224, batch=32, project=RESULTS, name=f"eval_online_{label.split()[0]}{label.split()[-1].strip('()b=')}", exist_ok=True)
    own = m.val(data=OWN, split="test", imgsz=224, batch=32, project=RESULTS, name=f"eval_own_{label.split()[0]}{label.split()[-1].strip('()b=')}", exist_ok=True)
    rows.append({"run": label, "online test top1": round(online.top1, 4), "online test top5": round(online.top5, 4),
                 "OWN photos top1": round(own.top1, 4), "OWN photos top5": round(own.top5, 4)})
table = pd.DataFrame(rows); table.to_csv(f"{RESULTS}/batch_size_comparison.csv", index=False)
table

Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n-cls summary (fused): 47 layers, 1,532,429 parameters, 0 gradients, 3.2 GFLOPs
train: /content/vegetable_dataset/train... found 5000 images in 5/5 classes ✅ 
val: /content/vegetable_dataset/val... found 1000 images in 5/5 classes ✅ 
test: /content/vegetable_dataset/test... found 1000 images in 5/5 classes ✅ 
WARNING ⚠️ test: Slow image access detected (ping: 0.0±0.0 ms, read: 11.1±8.3 MB/s, size: 44.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
test: Scanning /content/vegetable_dataset/test... 1000 images, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 1.2Kit/s 0.8s
test: New cache created: /content/vegetable_dataset/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 32/32 15.1it/s 2.1s
                   all          1          1
Speed: 0.1ms preprocess, 1.1ms inference, 0.0ms loss, 0

,run,online test top1,online test top5,OWN photos top1,OWN photos top5
0,baseline (b=16),1.0,1.0,0.7333,1.0
1,batch 8,1.0,1.0,0.7917,1.0
2,batch 32,1.0,1.0,0.7500,1.0
3,batch 64,1.0,1.0,0.7500,1.0


## 5. Per-class accuracy on own photos + list of mistakes (for the report)

In [ ]:
from collections import defaultdict
for label, w in runs.items():
    m = YOLO(w); correct = defaultdict(int); total = defaultdict(int); wrong = []
    for c in CLASSES:
        for f in sorted(glob.glob(f"{OWN}/test/{c}/*")):
            r = m.predict(f, imgsz=224, verbose=False)[0]
            pred = r.names[r.probs.top1]; total[c] += 1
            if pred == c: correct[c] += 1
            else: wrong.append((os.path.basename(f), c, pred, round(float(r.probs.top1conf), 2)))
    print(f"\n=== {label} ===")
    print({c: f"{correct[c]}/{total[c]}" for c in CLASSES})
    for x in wrong: print("  WRONG:", x[0], "| true:", x[1], "| predicted:", x[2], "| conf:", x[3])


=== baseline (b=16) ===
{'Broccoli': '24/24', 'Capsicum': '20/24', 'Carrot': '14/24', 'Potato': '15/24', 'Tomato': '15/24'}
  WRONG: capsicum_antony_003.jpg | true: Capsicum | predicted: Carrot | conf: 0.91
  WRONG: capsicum_antony_004.jpg | true: Capsicum | predicted: Broccoli | conf: 0.55
  WRONG: capsicum_sumerah_005.jpg | true: Capsicum | predicted: Broccoli | conf: 0.85
  WRONG: capsicum_sumerah_006.jpg | true: Capsicum | predicted: Potato | conf: 0.74
  WRONG: carrot_alyssa_001.jpg | true: Carrot | predicted: Tomato | conf: 1.0
  WRONG: carrot_alyssa_002.jpg | true: Carrot | predicted: Broccoli | conf: 0.43
  WRONG: carrot_alyssa_005.jpg | true: Carrot | predicted: Tomato | conf: 0.99
  WRONG: carrot_grace_001.jpg | true: Carrot | predicted: Broccoli | conf: 0.98
  WRONG: carrot_grace_002.jpg | true: Carrot | predicted: Broccoli | conf: 0.99
  WRONG: carrot_grace_003.jpg | true: Carrot | predicted: Broccoli | conf: 0.94
  WRONG: carrot_grace_004.jpg | true: Carrot | predicted: B

## 6. Where your report figures are (in Drive → `training_results/`)
- `batch8/`, `batch32/`, `batch64/` → `results.png` (training curves), `confusion_matrix_normalized.png`, `results.csv`
- `eval_own_*/` → confusion matrix on our own photos (most useful figure for the shortcomings section)
- `batch_size_comparison.csv` → the comparison table